오디오 파일, 라벨링 텍스트 가공

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#켤때마다 실행
!pip install --upgrade pip
!pip install transformers datasets evaluate jiwer librosa soundfile accelerate -U

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

# 학습할 기본 모델 지정 (base 사이즈)
model_name_or_path = "openai/whisper-base"

# 오디오 소리를 수치로 변환
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_or_path)

# 글자를 AI가 이해하는 숫자 변환
tokenizer = WhisperTokenizer.from_pretrained(model_name_or_path, language="Korean", task="transcribe")

# 위 기능들 묶음처리
processor = WhisperProcessor.from_pretrained(model_name_or_path, language="Korean", task="transcribe")

print("Whisper 전처리 도구(Feature Extractor, Tokenizer) 로드 완료")

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("AI용 데이터 최종 변환 중...")

ready_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)
print("AI가 인지 가능한 최종 데이터 변환 완료")

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
import numpy as np

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# WER: 단어 에러율 - 낮을수록 좋음
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("포장 & 채점 세팅 완료")

In [ ]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

print("Whisper AI Model 다운로드 중...")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

# 한국어 학습을 위한 모델 필수 세팅
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-mini-test",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    max_steps=50,
    fp16=True,
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    eval_steps=25,
    logging_steps=10,
    save_steps=25,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=ready_dataset,
    eval_dataset=ready_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("==Fine-tuning 시작==")
trainer.train()